In [2]:
import time
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, GridSearchCV, cross_validate
from xgboost import XGBRegressor

In [3]:
X_train = pd.read_csv("../data/processed/X_train.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze("columns")

preprocessor = joblib.load("../models/preprocessor.pkl")

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (1239, 92)
y_train shape: (1239,)


In [4]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = "neg_root_mean_squared_error"

In [5]:
ridge_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", Ridge())
])

In [6]:
baseline_scores = cross_validate(
    ridge_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2"
    },
    n_jobs=2,
    error_score="raise"
)

print("Ridge baseline CV RMSE:",
      -baseline_scores["test_rmse"].mean())

print("Ridge baseline CV R²:",
      baseline_scores["test_r2"].mean())

Ridge baseline CV RMSE: 0.11953584913309045
Ridge baseline CV R²: 0.9085470992275244


In [7]:
ridge_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid={
        "model__alpha": np.logspace(-3, 3, 61)
    },
    scoring=scoring,
    cv=cv,
    n_jobs=2,
    refit=True,
    error_score="raise",
    return_train_score=True
)

start_time = time.time()

ridge_search.fit(X_train, y_train)

elapsed = time.time() - start_time

print("Best alpha:",
      ridge_search.best_params_["model__alpha"])

print("Best CV RMSE:",
      -ridge_search.best_score_)

print("Tuning time:",
      round(elapsed, 2), "seconds")

Best alpha: 19.952623149688787
Best CV RMSE: 0.11421191908595789
Tuning time: 8.09 seconds


In [8]:
best_ridge_pipeline = ridge_search.best_estimator_

best_index = ridge_search.best_index_

print("Best Ridge CV RMSE:",
      -ridge_search.cv_results_["mean_test_score"][best_index])

print("Best Ridge CV standard deviation:",
      ridge_search.cv_results_["std_test_score"][best_index])

Best Ridge CV RMSE: 0.11421191908595789
Best Ridge CV standard deviation: 0.008367168099175225


In [9]:
ridge_results = pd.DataFrame({
    "alpha": ridge_search.cv_results_["param_model__alpha"].astype(float),
    "rmse": -ridge_search.cv_results_["mean_test_score"],
    "std": ridge_search.cv_results_["std_test_score"]
}).sort_values("rmse")

print(ridge_results.head(10))

        alpha      rmse       std
43  19.952623  0.114212  0.008367
42  15.848932  0.114232  0.008307
44  25.118864  0.114279  0.008416
41  12.589254  0.114337  0.008237
45  31.622777  0.114434  0.008453
40  10.000000  0.114522  0.008163
46  39.810717  0.114673  0.008480
39   7.943282  0.114783  0.008089
47  50.118723  0.114994  0.008498
38   6.309573  0.115112  0.008023


In [10]:
old_rmse = 0.11953331677383647
new_rmse = -ridge_search.best_score_

print("Previous Ridge CV RMSE:", old_rmse)
print("New Ridge CV RMSE:", new_rmse)
print("Improvement:", old_rmse - new_rmse)

Previous Ridge CV RMSE: 0.11953331677383647
New Ridge CV RMSE: 0.11421191908595789
Improvement: 0.005321397687878579
